# 01 - Explore the parsed BRSR filings

A first look at the Parquet tables written by `python -m ingestion.parse_xbrl`
(`data/processed/parsed/<table>/reporting_year_label=<year>/<filing_id>.parquet`).

Everything is queried with DuckDB straight from the Parquet files, with no warehouse needed.
Values are **raw strings exactly as filed**. Nothing is cast or cleaned here; that is the job of
the dbt staging models.

> The real filings are not in git (NSE terms of use). This notebook only works on a machine
> where the files were downloaded by hand and parsed. **Clear all outputs before committing.**

## Setup

Create one DuckDB view per table. `hive_partitioning = true` turns the folder name
`reporting_year_label=2025-2026` into a normal column.

In [ ]:
from pathlib import Path

import duckdb

ROOT = Path.cwd() if (Path.cwd() / "ingestion").is_dir() else Path.cwd().parent
PARSED = ROOT / "data" / "processed" / "parsed"

con = duckdb.connect()
for table in ["filings", "contexts", "units", "facts", "text_facts"]:
    pattern = (PARSED / table).as_posix() + "/*/*.parquet"
    con.execute(
        f"create or replace view {table} as "
        f"select * from read_parquet('{pattern}', hive_partitioning = true)"
    )
con.sql("select count(*) as filings, sum(n_facts) as facts from filings")

## 1. Facts per company

How many facts each company filed in each reporting year (numeric + short text + long text).
A company with far fewer facts than its peers may have left sections blank. A missing year
means the filing was not received.

In [ ]:
con.sql("""
    pivot (select symbol, reporting_year_label, n_facts from filings)
    on reporting_year_label using sum(n_facts)
    order by symbol
""").df()

## 2. HDFC Bank: Scope 1, 2 and 3 emissions across the four years

Every filing carries the current year **and** the prior year, so each year shows two rows per
concept: one for each period. Comparing a year's prior-year row with the previous filing's
current-year row is how restatements will be spotted later.

`dimension_key` is empty for the company-level total. `raw_value` is the string as filed
(NULL when the fact is nil).

In [ ]:
con.sql("""
    select fl.reporting_year_label, f.concept, f.context_ref,
           c.start_date, c.end_date, c.dimension_key, u.unit_label, f.raw_value, f.is_nil
    from facts f
    join filings fl using (filing_id)
    join contexts c on c.filing_id = f.filing_id and c.context_id = f.context_ref
    left join units u on u.filing_id = f.filing_id and u.unit_id = f.unit_ref
    where fl.symbol = 'HDFCBANK'
      and f.concept in ('TotalScope1Emissions', 'TotalScope2Emissions', 'TotalScope3Emissions')
    order by f.concept, fl.reporting_year_label, c.end_date
""").df()

## 3. The 10 most common dimension axes

A dimension (axis) splits a number into parts, for example by gender or by employees vs
workers. This counts how many contexts use each axis across all filings. `dimensions` is a JSON
list, so it is unnested first.

In [ ]:
con.sql("""
    with dims as (
        select unnest(from_json(dimensions, '[{"axis": "VARCHAR", "member": "VARCHAR",
                                               "member_type": "VARCHAR"}]')) as d
        from contexts
    )
    select d.axis, d.member_type, count(*) as n_contexts
    from dims
    group by all
    order by n_contexts desc
    limit 10
""").df()

## 4. Units

Every unit used in the filings, with how many filings use it. Units built with `xbrli:divide`
show up as `numerator/denominator` (e.g. `tCO2e/INR` for emissions per rupee of turnover).

In [ ]:
con.sql("""
    select unit_label, numerator_measures, denominator_measures,
           count(distinct filing_id) as n_filings
    from units
    group by all
    order by n_filings desc, unit_label
""").df()

## 5. Non-standard reporting periods

Indian companies normally report April-March (12 months). This lists filings whose main
reporting period is not 12 months. Their figures are not directly comparable with a normal year
and need to be annualised or flagged.

In [ ]:
con.sql("""
    select symbol, reporting_year_label, main_context_id, period_start, period_end,
           period_days, period_months
    from filings
    where is_non_standard_period is distinct from false
    order by symbol, reporting_year_label
""").df()